# Island Conservation Across Mammalian Phylogeny

Analysis of aligned islands across 10 mammalian species to identify:
- **Conserved cores**: islands found across most/all species with consistently low MMD
- **Phylogenetic decay**: islands whose MMD increases with evolutionary distance from human
- **Aggregate statistics**: MMD distributions, island lengths, detection rates per species
- **Phylo-sorted heatmaps**: visual summary of conservation patterns

In [ ]:
import sys
from pathlib import Path
from collections import defaultdict

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import seaborn as sns
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = 'white'
sns.set_style('whitegrid')

RESULTS_DIR = Path("../preprint_results")
ANNOTATION_DIR = Path("../input_data/reference_annotation")
FIGURES_DIR = Path("raw_plots")


## 1. Species metadata & phylogenetic ordering

In [ ]:
SPECIES_META = {
    "rheMac10":  {"name": "Rhesus macaque",   "order": "Primates",         "clade": "Euarchontoglires", "div_mya": 25},
    "mm39":      {"name": "Mouse",            "order": "Rodentia",         "clade": "Euarchontoglires", "div_mya": 90},
    "rn7":       {"name": "Rat",              "order": "Rodentia",         "clade": "Euarchontoglires", "div_mya": 90},
    "bosTau9":   {"name": "Cow",              "order": "Artiodactyla",     "clade": "Laurasiatheria",  "div_mya": 94},
    "susScr11":  {"name": "Pig",              "order": "Artiodactyla",     "clade": "Laurasiatheria",  "div_mya": 94},
    "equCab3":   {"name": "Horse",            "order": "Perissodactyla",   "clade": "Laurasiatheria",  "div_mya": 94},
    "felCat9":   {"name": "Cat",              "order": "Carnivora",        "clade": "Laurasiatheria",  "div_mya": 94},
    "eriEur2":   {"name": "Hedgehog",         "order": "Eulipotyphla",     "clade": "Laurasiatheria",  "div_mya": 94},
    "dasNov3":   {"name": "Armadillo",        "order": "Cingulata",        "clade": "Xenarthra",       "div_mya": 105},
    "monDom5":   {"name": "Opossum",          "order": "Didelphimorphia",  "clade": "Marsupialia",     "div_mya": 180},
}

PHYLO_ORDER = ["rheMac10", "mm39", "rn7", "bosTau9", "susScr11",
               "equCab3", "felCat9", "eriEur2", "dasNov3", "monDom5"]

species_labels = {k: f"{v['name']} ({k})" for k, v in SPECIES_META.items()}
phylo_labels = [species_labels[s] for s in PHYLO_ORDER]
div_times = [SPECIES_META[s]["div_mya"] for s in PHYLO_ORDER]

print(f"Species count: {len(PHYLO_ORDER)}")
for s in PHYLO_ORDER:
    m = SPECIES_META[s]
    print(f"  {m['name']:20s} ({s:10s})  {m['clade']:20s}  ~{m['div_mya']} Mya")

## 2. Load all island alignment results

In [ ]:
frames = []
for sp in PHYLO_ORDER:
    tsv_path = RESULTS_DIR / f"hg38_vs_{sp}" / "island_alignment_results.tsv"
    df = pd.read_csv(tsv_path, sep="\t")
    df["species"] = sp
    frames.append(df)

all_islands = pd.concat(frames, ignore_index=True)
print(f"Total rows loaded: {len(all_islands):,}")
print(f"Columns: {list(all_islands.columns)}")
all_islands.head(3)

In [ ]:
# Unique key for each reference island: gene_id + ref_island number
all_islands["island_key"] = all_islands["gene_id"] + "_" + all_islands["ref_island"]

# Add species metadata
all_islands["species_name"] = all_islands["species"].map(lambda s: SPECIES_META[s]["name"])
all_islands["div_mya"] = all_islands["species"].map(lambda s: SPECIES_META[s]["div_mya"])
all_islands["clade"] = all_islands["species"].map(lambda s: SPECIES_META[s]["clade"])

print(f"Unique reference islands: {all_islands['island_key'].nunique():,}")
print(f"Unique genes: {all_islands['gene_id'].nunique():,}")

### 2.1 Merge biotype information

In [ ]:
meta_path = RESULTS_DIR / "hg38_vs_rheMac10" / "reference_union_transcripts_metadata.tsv"
biotype_df = pd.read_csv(meta_path, sep="\t")
biotype_map = biotype_df.set_index("transcript_id")["biotype"].to_dict()

all_islands["biotype"] = all_islands["gene_id"].map(biotype_map)
print("Biotype distribution (top 10):")
print(all_islands["biotype"].value_counts().head(10))

### 2.2 Load reference island strand info from BED

In [ ]:
bed_path = RESULTS_DIR / "hg38_vs_rheMac10" / "query_annotation" / "reference_islands.bed"
bed_cols = ["chrom", "start", "end", "name", "score", "strand",
            "thickStart", "thickEnd", "rgb", "blockCount", "blockSizes", "blockStarts"]
ref_islands_bed = pd.read_csv(bed_path, sep="\t", header=None, names=bed_cols)

# Parse island key from BED name: U_ENSG00000099869.8_island_0 -> U_ENSG00000099869.8_R0
def bed_name_to_key(name):
    parts = name.rsplit("_island_", 1)
    return f"{parts[0]}_R{parts[1]}"

ref_islands_bed["island_key"] = ref_islands_bed["name"].apply(bed_name_to_key)
ref_islands_bed["island_len_bed"] = ref_islands_bed["end"] - ref_islands_bed["start"]

strand_map = ref_islands_bed.set_index("island_key")["strand"].to_dict()
island_len_map = ref_islands_bed.set_index("island_key")["island_len_bed"].to_dict()

all_islands["ref_strand"] = all_islands["island_key"].map(strand_map)
all_islands["ref_island_len_full"] = all_islands["island_key"].map(island_len_map)

print(f"Strand coverage: {all_islands['ref_strand'].notna().mean():.1%}")
print(f"Reference islands in BED: {len(ref_islands_bed):,}")
ref_islands_bed.head(3)

### 2.3 Optionally load gene names

In [ ]:
all_islands["gene_id_bare"] = all_islands["gene_id"].str.replace("^U_", "", regex=True).str.split(".").str[0]

# Prefer mart_export.txt (BioMart) — it covers XIST, XACT, etc. that hg38_gene_names.txt misses
mart_path = ANNOTATION_DIR / "mart_export.txt"
gene_names_path = ANNOTATION_DIR / "hg38_gene_names.txt"

if mart_path.exists():
    mart = pd.read_csv(mart_path, sep="\t")
    gn_map = (
        mart[mart["Gene name"].notna() & (mart["Gene name"] != "")]
        .drop_duplicates(subset="Gene stable ID")
        .set_index("Gene stable ID")["Gene name"]
        .to_dict()
    )
    print(f"Loaded {len(gn_map):,} gene names from mart_export.txt")
elif gene_names_path.exists():
    gn = pd.read_csv(gene_names_path, sep="\t")
    gn_map = gn.drop_duplicates(subset="Gene stable ID").set_index("Gene stable ID")["Gene name"].to_dict()
    print(f"Loaded {len(gn_map):,} gene names from hg38_gene_names.txt")
else:
    gn_map = {}
    print("No gene name files found.")

all_islands["gene_name"] = all_islands["gene_id_bare"].map(gn_map)
all_islands["gene_name"] = all_islands["gene_name"].fillna(all_islands["gene_id_bare"])
named = (all_islands["gene_name"] != all_islands["gene_id_bare"]).mean()
print(f"Genes with a proper symbol: {named:.1%}  (rest fall back to ENSG ID)")

### 2.4 Cluster aligned intervals into conserved cores

The aligned coordinates on the reference can drift between species (e.g., R1 aligns
to 100-200 in mouse but 102-198 in rat). Instead of matching by ref_island label,
we cluster aligned reference intervals **within each gene** across all species using
overlap on hg38 coordinates. Two intervals from different species are placed in the
same "conserved core" if they share ≥ 50 % reciprocal overlap (relative to the
shorter interval) and ≥ 20 bp absolute overlap.

In [ ]:
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

MIN_OVERLAP_FRAC = 0.50
MIN_OVERLAP_BP = 20

def _cluster_intervals(starts, ends, min_frac=MIN_OVERLAP_FRAC, min_bp=MIN_OVERLAP_BP):
    """Assign connected-component labels to a set of intervals.

    Two intervals are linked if their overlap is >= min_bp **and**
    >= min_frac of the shorter interval's length.
    """
    n = len(starts)
    if n <= 1:
        return np.zeros(n, dtype=int)

    rows, cols = [], []
    for i in range(n):
        for j in range(i + 1, n):
            ovl = max(0, min(ends[i], ends[j]) - max(starts[i], starts[j]))
            if ovl < min_bp:
                continue
            shorter = min(ends[i] - starts[i], ends[j] - starts[j])
            if shorter > 0 and ovl / shorter >= min_frac:
                rows += [i, j]
                cols += [j, i]

    if not rows:
        return np.arange(n, dtype=int)

    adj = csr_matrix((np.ones(len(rows), dtype=np.int8), (rows, cols)), shape=(n, n))
    _, labels = connected_components(adj, directed=False)
    return labels


def assign_core_ids(df):
    """For each gene, cluster aligned ref intervals into conserved cores.

    Adds a 'core_id' column of the form  <gene_id>_C<cluster_number>.
    """
    core_ids = pd.Series(index=df.index, dtype=str)

    for gene_id, gdf in df.groupby("gene_id"):
        # Deduplicate intervals: unique (ref_start, ref_end) per row
        starts = gdf["ref_start"].values
        ends = gdf["ref_end"].values
        labels = _cluster_intervals(starts, ends)
        core_ids.loc[gdf.index] = [f"{gene_id}_C{l}" for l in labels]

    return core_ids


all_islands["orig_island_key"] = all_islands["island_key"]
all_islands["core_id"] = assign_core_ids(all_islands)

n_orig = all_islands["orig_island_key"].nunique()
n_cores = all_islands["core_id"].nunique()
print(f"Original unique island keys:  {n_orig:,}")
print(f"Conserved cores after merge:  {n_cores:,}")
print(f"Reduction: {n_orig - n_cores:,} keys collapsed ({(1 - n_cores/n_orig)*100:.1f}%)")

In [ ]:
# Diagnostics: how much did cores differ from original island keys?
merge_stats = (
    all_islands.groupby("core_id")["orig_island_key"]
    .nunique()
    .rename("n_orig_keys")
)
print("Original island keys per conserved core:")
print(merge_stats.value_counts().sort_index().to_string())

# Show examples of merged cores (where > 1 original key collapsed)
merged_examples = merge_stats[merge_stats > 1]
if len(merged_examples) > 0:
    print(f"\nCores that merged multiple original keys: {len(merged_examples):,}")
    for cid in merged_examples.head(5).index:
        sub = all_islands[all_islands["core_id"] == cid][
            ["gene_id", "species", "ref_island", "ref_start", "ref_end", "orig_island_key"]
        ].drop_duplicates(subset=["species", "orig_island_key"]).sort_values(["orig_island_key", "species"])
        print(f"\n  Core {cid}:")
        print(sub[["species", "ref_island", "ref_start", "ref_end"]].to_string(index=False))
else:
    print("\nNo cores merged multiple original keys (coordinates were consistent).")

## 3. Aggregate statistics per species

In [ ]:
species_stats = []
for sp in PHYLO_ORDER:
    sub = all_islands[all_islands["species"] == sp]
    species_stats.append({
        "species": sp,
        "name": SPECIES_META[sp]["name"],
        "div_mya": SPECIES_META[sp]["div_mya"],
        "n_matches": len(sub),
        "n_genes": sub["gene_id"].nunique(),
        "n_ref_islands": sub["island_key"].nunique(),
        "median_mmd": sub["diag_mmd"].median(),
        "mean_mmd": sub["diag_mmd"].mean(),
        "median_ref_len": sub["ref_len"].median(),
        "median_query_len": sub["query_len"].median(),
        "median_len_ratio": (sub["query_len"] / sub["ref_len"]).median(),
    })

stats_df = pd.DataFrame(species_stats)
stats_df

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

ax = axes[0, 0]
ax.barh(range(len(stats_df)), stats_df["n_matches"], color="steelblue")
ax.set_yticks(range(len(stats_df)))
ax.set_yticklabels([f"{r['name']} ({r['div_mya']} My)" for _, r in stats_df.iterrows()])
ax.set_xlabel("Number of island matches")
ax.set_title("Island matches per species")
ax.invert_yaxis()

ax = axes[0, 1]
ax.barh(range(len(stats_df)), stats_df["n_ref_islands"], color="darkorange")
ax.set_yticks(range(len(stats_df)))
ax.set_yticklabels([f"{r['name']}" for _, r in stats_df.iterrows()])
ax.set_xlabel("Unique reference islands matched")
ax.set_title("Ref. islands detected per species")
ax.invert_yaxis()

ax = axes[1, 0]
ax.scatter(stats_df["div_mya"], stats_df["median_mmd"], s=80, zorder=3, color="crimson")
for _, r in stats_df.iterrows():
    ax.annotate(r["name"], (r["div_mya"], r["median_mmd"]),
                fontsize=7, ha="left", va="bottom", xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("Divergence time (Mya)")
ax.set_ylabel("Median MMD")
ax.set_title("Median MMD vs. divergence time")

ax = axes[1, 1]
ax.scatter(stats_df["div_mya"], stats_df["median_len_ratio"], s=80, zorder=3, color="seagreen")
for _, r in stats_df.iterrows():
    ax.annotate(r["name"], (r["div_mya"], r["median_len_ratio"]),
                fontsize=7, ha="left", va="bottom", xytext=(3, 3), textcoords="offset points")
ax.axhline(1.0, ls="--", color="gray", alpha=0.5)
ax.set_xlabel("Divergence time (Mya)")
ax.set_ylabel("Median query/ref length ratio")
ax.set_title("Length ratio vs. divergence time")

plt.tight_layout()
plt.show()

## 4. MMD distributions per species (phylo-sorted)

In [ ]:
species_order_labels = [species_labels[s] for s in PHYLO_ORDER]
all_islands["species_label"] = all_islands["species"].map(species_labels)

stats = all_islands.groupby("species")["diag_mmd"].agg(
    median="median", q25=lambda x: x.quantile(0.25), q75=lambda x: x.quantile(0.75)
).reindex(PHYLO_ORDER)

div_times = [SPECIES_META[s]["div_mya"] for s in PHYLO_ORDER]
labels = [f"{species_labels[s]}\n({SPECIES_META[s]['div_mya']} My)"
          for s in PHYLO_ORDER]

fig, ax = plt.subplots(figsize=(8, 4.5))

x = np.arange(len(PHYLO_ORDER))
yerr_lo = stats["median"] - stats["q25"]
yerr_hi = stats["q75"] - stats["median"]

ax.errorbar(x, stats["median"], yerr=[yerr_lo, yerr_hi],
            fmt="o-", color="#5B4B8A", capsize=4, capthick=1.2,
            markersize=7, markerfacecolor="#5B4B8A", markeredgecolor="white",
            markeredgewidth=0.8, linewidth=1.8, ecolor="#8B7BBF", elinewidth=1.2)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel("MMD", fontsize=11)
ax.set_title("Island alignment MMD across species (median \u00b1 IQR)",
             fontsize=11, fontweight="bold")
ax.set_ylim(bottom=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "MMD_per_species_phylo.pdf",
            format="pdf", bbox_inches="tight")
plt.show()
print(f"Saved: {FIGURES_DIR / 'MMD_per_species_phylo.pdf'}")


In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(
    data=all_islands, x="species_label", y="ref_len",
    order=species_order_labels, cut=0, inner="quartile",
    palette="Blues", ax=ax, scale="width"
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=8)
ax.set_xlabel("")
ax.set_ylabel("Aligned island ref length (bp)")
ax.set_title("Reference island length distribution per species")
ax.set_ylim(0, all_islands["ref_len"].quantile(0.99))
plt.tight_layout()
plt.show()

## 5. Island conservation: how many species detect each island?

In [ ]:
# Use core_id (overlap-based clustering) instead of island_key for conservation analysis.
# For each conserved core, take best (lowest) MMD match per species.
best_per_species = (
    all_islands
    .sort_values("diag_mmd")
    .drop_duplicates(subset=["core_id", "species"], keep="first")
)
print(f"Best-per-species rows: {len(best_per_species):,}")

conservation = best_per_species.groupby("core_id")["species"].nunique().rename("n_species")
conservation_with_mmd = (
    best_per_species.groupby("core_id")
    .agg(n_species=("species", "nunique"),
         mean_mmd=("diag_mmd", "mean"),
         median_mmd=("diag_mmd", "median"),
         min_mmd=("diag_mmd", "min"),
         max_mmd=("diag_mmd", "max"),
         mean_ref_len=("ref_len", "mean"))
    .reset_index()
)

# Add gene_id and biotype
key_to_gene = all_islands.drop_duplicates("core_id").set_index("core_id")[["gene_id", "biotype", "gene_name"]]
conservation_with_mmd = conservation_with_mmd.join(key_to_gene, on="core_id")

print(f"\nConserved-core conservation distribution:")
print(conservation.value_counts().sort_index())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
counts = conservation.value_counts().sort_index()
ax.bar(counts.index, counts.values, color="teal", edgecolor="white")
ax.set_xlabel("Number of species with match")
ax.set_ylabel("Number of conserved cores")
ax.set_title("Conserved core conservation spectrum")
ax.set_xticks(range(1, 12))

ax = axes[1]
ax.scatter(conservation_with_mmd["n_species"], conservation_with_mmd["mean_mmd"],
           alpha=0.1, s=5, color="firebrick")
medians = conservation_with_mmd.groupby("n_species")["mean_mmd"].median()
ax.plot(medians.index, medians.values, "ko-", ms=6, lw=2, label="Median")
ax.set_xlabel("Number of species with match")
ax.set_ylabel("Mean MMD across species")
ax.set_title("Conservation breadth vs. MMD")
ax.legend()

plt.tight_layout()
plt.show()

### Core reproducibility (cumulative)

Fraction of conserved cores detected in **at least k** species.
Shows that conservation is not anecdotal but a bulk property.

In [ ]:
total_cores = len(conservation)
k_vals = np.arange(1, 11)
cumulative = np.array([( conservation >= k ).sum() / total_cores for k in k_vals])

fig, ax = plt.subplots(figsize=(6, 4.5))

ax.bar(k_vals, cumulative, color="#5B4B8A", edgecolor="white", width=0.7)

for k, frac in zip(k_vals, cumulative):
    n = int((conservation >= k).sum())
    ax.text(k, frac + 0.01, f"{n:,}", ha="center", va="bottom", fontsize=7)

ax.set_xticks(k_vals)
ax.set_xticklabels([f"\u2265{k}" for k in k_vals], fontsize=9)
ax.set_xlabel("Number of species", fontsize=11)
ax.set_ylabel("Fraction of conserved cores", fontsize=11)
ax.set_title("Core reproducibility across species",
             fontsize=11, fontweight="bold")
ax.set_ylim(0, 1.08)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
fig.savefig(FIGURES_DIR / "core_reproducibility_cumulative.pdf",
            format="pdf", bbox_inches="tight")
plt.show()
print(f"Total cores: {total_cores:,}")
print(f"Saved: {FIGURES_DIR / 'core_reproducibility_cumulative.pdf'}")


### 5.1 Conservation spectrum by biotype

In [ ]:
top_biotypes = conservation_with_mmd["biotype"].value_counts().head(6).index.tolist()
sub = conservation_with_mmd[conservation_with_mmd["biotype"].isin(top_biotypes)]

fig, ax = plt.subplots(figsize=(10, 5))
for bt in top_biotypes:
    btsub = sub[sub["biotype"] == bt]
    counts = btsub["n_species"].value_counts().sort_index()
    fracs = counts / counts.sum()
    ax.plot(fracs.index, fracs.values, "o-", label=bt, ms=5)

ax.set_xlabel("Number of species with match")
ax.set_ylabel("Fraction of islands")
ax.set_title("Conservation spectrum by biotype")
ax.set_xticks(range(1, 12))
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 6. Case studies: iconic lncRNAs

Per-gene deep dives on MALAT1, NEAT1, XIST, XACT, MIAT, NORAD, HOTAIR, H19, TUG1,
RMST, LINC-PINT, FIRRE, PVT1, DDX25-AS1, HYOU1-AS1, KCNQ1OT1, MEG3.
For each gene we show a **core × species heatmap** (MMD colour) and a matching
**aligned-length heatmap**, both phylo-sorted.

In [ ]:
CASE_STUDY_GENES = {
    "MALAT1":     "ENSG00000251562",
    "NEAT1":      "ENSG00000245532",
    "XIST":       "ENSG00000229807",
    "XACT":       "ENSG00000241743",
    "MIAT":       "ENSG00000225783",
    "NORAD":      "ENSG00000260032",
    "HOTAIR":     "ENSG00000228630",
    "H19":        "ENSG00000288237",
    "TUG1":       "ENSG00000253352",
    "RMST":       "ENSG00000255794",
    "LINC-PINT":  "ENSG00000231721",
    "FIRRE":      "ENSG00000213468",
    "PVT1":       "ENSG00000249859",
    "DDX25-AS1":  "ENSG00000255027",
    "HYOU1-AS1":  "ENSG00000255114",
    "KCNQ1OT1":   "ENSG00000269821",
    "MEG3":       "ENSG00000214548",
}

MMD_VMIN, MMD_VMAX = 0, 0.25
MMD_CMAP = plt.cm.RdYlGn_r.copy()
MMD_CMAP.set_bad(color="#e0e0e0")

def plot_gene_case_study(gene_symbol, ensg_id, best_df, all_df, phylo_order, species_meta):
    """Plot MMD heatmap for one gene (species=Y, cores=X). Save PDF to raw_plots/."""

    mask = all_df["gene_id_bare"] == ensg_id
    gene_df = all_df[mask]
    if gene_df.empty:
        print(f"  {gene_symbol} ({ensg_id}): no island data in results.")
        print(f"  \u2192 This gene has no detectable conserved exonic islands.")
        return

    gene_ids = gene_df["gene_id"].unique()
    best_mask = best_df["gene_id"].isin(gene_ids)
    gene_best = best_df[best_mask].copy()

    cores = sorted(gene_best["core_id"].unique())
    n_cores = len(cores)
    species_names = [species_meta[s]["name"] for s in phylo_order]

    mmd_pivot = (
        gene_best.pivot_table(index="species", columns="core_id",
                              values="diag_mmd", aggfunc="min")
        .reindex(index=phylo_order, columns=cores)
    )

    col_labels = [f"C{c.rsplit('_C', 1)[-1]}" for c in cores]

    n_species = len(phylo_order)

    fig, ax = plt.subplots(figsize=(max(5, n_cores * 0.8 + 3),
                                    max(4, n_species * 0.4 + 1.5)))

    mmd_arr = mmd_pivot.values.copy()
    im = ax.imshow(np.where(np.isnan(mmd_arr), np.nan, mmd_arr),
                    aspect="auto", cmap=MMD_CMAP, vmin=MMD_VMIN, vmax=MMD_VMAX)
    ax.set_xticks(range(n_cores))
    ax.set_xticklabels(col_labels, fontsize=8)
    ax.set_yticks(range(n_species))
    ax.set_yticklabels(species_names, fontsize=8)
    ax.set_title(f"{gene_symbol}  ({ensg_id})", fontweight="bold", fontsize=11)
    plt.colorbar(im, ax=ax, shrink=0.6, label="MMD")

    for i in range(n_species):
        for j in range(n_cores):
            v = mmd_arr[i, j]
            if not np.isnan(v):
                ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                        fontsize=6, color="white" if v > 0.15 else "black")
            else:
                ax.text(j, i, "\u2014", ha="center", va="center",
                        fontsize=7, color="#999999")

    plt.tight_layout()
    safe_name = gene_symbol.replace("-", "_")
    fig.savefig(FIGURES_DIR / f"{safe_name}_PHYLO.pdf",
               format="pdf", bbox_inches="tight")
    plt.show()
    print(f"  Saved: {FIGURES_DIR / f'{safe_name}_PHYLO.pdf'}")

    n_species_found = mmd_pivot.notna().any(axis=1).sum()
    mean_mmd = np.nanmean(mmd_arr)
    print(f"  Cores: {n_cores} | Species with \u22651 match: {n_species_found}/{len(phylo_order)}"
          f" | Global mean MMD: {mean_mmd:.4f}")

    sp_summary = gene_best.groupby("species").agg(
        n_cores=("core_id", "nunique"),
        median_mmd=("diag_mmd", "median"),
    ).reindex(phylo_order)
    sp_summary["name"] = [species_meta[s]["name"] for s in phylo_order]
    sp_summary["div_mya"] = [species_meta[s]["div_mya"] for s in phylo_order]
    print(sp_summary[["name", "div_mya", "n_cores", "median_mmd"]].to_string())


In [ ]:
print("=" * 60)
print("MALAT1 — Metastasis Associated Lung Adenocarcinoma Transcript 1")
print("Highly conserved single-exon lncRNA, nuclear speckle marker")
print("=" * 60)
plot_gene_case_study("MALAT1", "ENSG00000251562",
                     best_per_species, all_islands, PHYLO_ORDER, SPECIES_META)

### 6.1 NEAT1

In [ ]:
print("=" * 60)
print("NEAT1 — Nuclear Enriched Abundant Transcript 1")
print("Paraspeckle structural scaffold, multi-exon")
print("=" * 60)
plot_gene_case_study("NEAT1", "ENSG00000245532",
                     best_per_species, all_islands, PHYLO_ORDER, SPECIES_META)

In [ ]:
print("=" * 60)
print("XIST — X-Inactive Specific Transcript")
print("X-chromosome inactivation master regulator")
print("=" * 60)
plot_gene_case_study("XIST", "ENSG00000229807",
                     best_per_species, all_islands, PHYLO_ORDER, SPECIES_META)

In [ ]:
print("=" * 60)
print("XACT — X-Active Coating Transcript")
print("Primate-specific lncRNA coating the active X chromosome")
print("=" * 60)
plot_gene_case_study("XACT", "ENSG00000241743",
                     best_per_species, all_islands, PHYLO_ORDER, SPECIES_META)
print("\n→ XACT is expected to have no conserved cores outside primates.")
print("  The complete absence confirms its primate-specific origin.")

In [ ]:
print("=" * 60)
print("MIAT — Myocardial Infarction Associated Transcript")
print("Nuclear-retained lncRNA involved in neuronal/cardiac regulation")
print("=" * 60)
plot_gene_case_study("MIAT", "ENSG00000225783",
                     best_per_species, all_islands, PHYLO_ORDER, SPECIES_META)

In [ ]:
print("=" * 60)
print("NORAD — Non-coding RNA Activated by DNA Damage")
print("Pumilio-sequestering lncRNA, genome stability guardian")
print("=" * 60)
plot_gene_case_study("NORAD", "ENSG00000260032",
                     best_per_species, all_islands, PHYLO_ORDER, SPECIES_META)

In [ ]:
print("=" * 60)
print("HOTAIR — HOX Transcript Antisense Intergenic RNA")
print("Chromatin-remodelling lncRNA, PRC2 recruiter")
print("=" * 60)
plot_gene_case_study("HOTAIR", "ENSG00000228630",
                     best_per_species, all_islands, PHYLO_ORDER, SPECIES_META)

In [ ]:
print("=" * 60)
print("H19 — H19 Imprinted Maternally Expressed Transcript")
print("Imprinted lncRNA at IGF2 locus, tumour suppressor candidate")
print("=" * 60)
plot_gene_case_study("H19", "ENSG00000288237",
                     best_per_species, all_islands, PHYLO_ORDER, SPECIES_META)
print("\n→ H19: no exonic islands detected by the pipeline.")

In [ ]:
print("=" * 60)
print("TUG1 — Taurine Up-Regulated 1")
print("Transcriptional regulator lncRNA (classified protein_coding in this annotation)")
print("=" * 60)
plot_gene_case_study("TUG1", "ENSG00000253352",
                     best_per_species, all_islands, PHYLO_ORDER, SPECIES_META)
print("\n→ TUG1: no exonic islands detected. Note: classified as protein_coding"
      " in this GENCODE version, not lncRNA.")

In [ ]:
print("=" * 60)
print("RMST — Rhabdomyosarcoma 2-Associated Transcript")
print("Brain-enriched lncRNA, neuronal differentiation regulator")
print("=" * 60)
plot_gene_case_study("RMST", "ENSG00000255794",
                     best_per_species, all_islands, PHYLO_ORDER, SPECIES_META)

In [ ]:
print("=" * 60)
print("LINC-PINT — Long Intergenic Non-Protein Coding RNA, p53-Induced Transcript")
print("p53-regulated tumour suppressor lncRNA, PRC2 interactor")
print("=" * 60)
plot_gene_case_study("LINC-PINT", "ENSG00000231721",
                     best_per_species, all_islands, PHYLO_ORDER, SPECIES_META)

In [ ]:
print("=" * 60)
print("FIRRE — Firre Intergenic Repeating RNA Element")
print("X-linked lncRNA with trans-chromosomal interactions, repeat-rich architecture")
print("=" * 60)
plot_gene_case_study("FIRRE", "ENSG00000213468",
                     best_per_species, all_islands, PHYLO_ORDER, SPECIES_META)

In [ ]:
print("=" * 60)
print("PVT1 — Plasmacytoma Variant Translocation 1")
print("MYC-adjacent oncogenic lncRNA, miRNA sponge")
print("=" * 60)
plot_gene_case_study("PVT1", "ENSG00000249859",
                     best_per_species, all_islands, PHYLO_ORDER, SPECIES_META)

In [ ]:
print("=" * 60)
print("DDX25-AS1 — DDX25 Antisense RNA 1")
print("Antisense lncRNA overlapping DEAD-box helicase DDX25")
print("=" * 60)
plot_gene_case_study("DDX25-AS1", "ENSG00000255027",
                     best_per_species, all_islands, PHYLO_ORDER, SPECIES_META)

In [ ]:
print("=" * 60)
print("HYOU1-AS1 — HYOU1 Antisense RNA 1")
print("Antisense lncRNA at the hypoxia up-regulated 1 (HYOU1) locus")
print("=" * 60)
plot_gene_case_study("HYOU1-AS1", "ENSG00000255114",
                     best_per_species, all_islands, PHYLO_ORDER, SPECIES_META)

In [ ]:
print("=" * 60)
print("KCNQ1OT1 — KCNQ1 Opposite Strand/Antisense Transcript 1")
print("Imprinted antisense lncRNA, chromatin silencing at 11p15.5")
print("=" * 60)
plot_gene_case_study("KCNQ1OT1", "ENSG00000269821",
                     best_per_species, all_islands, PHYLO_ORDER, SPECIES_META)

In [ ]:
print("=" * 60)
print("MEG3 — Maternally Expressed Gene 3")
print("Imprinted tumour suppressor lncRNA, p53 activator")
print("=" * 60)
plot_gene_case_study("MEG3", "ENSG00000214548",
                     best_per_species, all_islands, PHYLO_ORDER, SPECIES_META)

## Transcript structure with conserved cores

For NORAD, PVT1, and FIRRE: exon blocks with conserved-core positions mapped below.

In [ ]:
import pandas as pd
import json

BED_PATH = RESULTS_DIR / "hg38_vs_mm39" / "reference_union_transcripts.bed"
REF_JSON = RESULTS_DIR / "hg38_vs_mm39" / "preprocessed_reference_data.json"

bed = pd.read_csv(BED_PATH, sep="\t", header=None,
    names=["chrom","start","end","name","score","strand",
           "thickStart","thickEnd","rgb","blockCount","blockSizes","blockStarts"])

with open(REF_JSON) as f:
    ref_data = json.load(f)

TRACK_GENES = [
    ("NORAD",  "U_ENSG00000260032.3"),
    ("PVT1",   "U_ENSG00000249859.15"),
    ("FIRRE",  "U_ENSG00000213468.8"),
]


def plot_transcript_cores(gene_symbol, gene_id, bed_df, ref_data,
                          best_df, min_island_len=72):
    ensg = gene_id.split(".")[0].replace("U_", "")
    tx = bed_df[bed_df["name"] == gene_id].iloc[0]
    tx_start = tx["start"]
    tx_end = tx["end"]
    is_minus = tx["strand"] == "-"

    sizes = [int(x) for x in tx["blockSizes"].strip(",").split(",") if x]
    starts = [int(x) for x in tx["blockStarts"].strip(",").split(",") if x]
    exons = [(tx_start + s, tx_start + s + sz) for s, sz in zip(starts, sizes)]

    # Islands (filtered + sorted, same as pipeline)
    all_islands_raw = ref_data[gene_id]["islands"]
    islands = sorted(
        [isl for isl in all_islands_raw if isl["end"] - isl["start"] >= min_island_len],
        key=lambda x: x["start"])

    # Map islands -> core_ids via best_per_species
    gene_best = best_df[best_df["gene_id"] == gene_id].copy()
    core_for_island = {}
    for _, r in gene_best.iterrows():
        key = (r["ref_chrom"], r["ref_start"], r["ref_end"])
        if key not in core_for_island:
            cnum = r["core_id"].rsplit("_C", 1)[-1]
            core_for_island[key] = f"C{cnum}"

    # Build labelled island list
    labelled = []
    for isl in islands:
        key = (isl["chrom"], isl["start"], isl["end"])
        label = core_for_island.get(key)
        if label is None:
            for k, v in core_for_island.items():
                if k[0] == isl["chrom"] and abs(k[1] - isl["start"]) < 50 and abs(k[2] - isl["end"]) < 50:
                    label = v
                    break
        labelled.append((isl["start"], isl["end"], label))

    # Flip coordinates for minus-strand genes so 5'->3' reads left-to-right
    if is_minus:
        def flip(s, e):
            return tx_end - e, tx_end - s
        exons = [flip(s, e) for s, e in exons]
        labelled = [(tx_end - e, tx_end - s, lbl) for s, e, lbl in labelled]
        plot_start, plot_end = 0, tx_end - tx_start
    else:
        plot_start, plot_end = tx_start, tx_end

    fig, ax = plt.subplots(figsize=(14, 1.8))

    exon_y = 0.6
    core_y = 0.2
    exon_h = 0.25
    core_h = 0.18

    # Intron line
    ax.plot([plot_start, plot_end], [exon_y, exon_y],
            color="#555555", linewidth=1, zorder=1)

    # Exon blocks
    for i, (es, ee) in enumerate(exons):
        ax.add_patch(plt.Rectangle(
            (es, exon_y - exon_h / 2), ee - es, exon_h,
            facecolor="#4A90D9", edgecolor="#2C5F8A", linewidth=0.8, zorder=3))
        mid = (es + ee) / 2
        if (ee - es) > (plot_end - plot_start) * 0.02:
            ax.text(mid, exon_y, f"E{i+1}", ha="center", va="center",
                    fontsize=6, color="white", fontweight="bold", zorder=4)

    # Core blocks
    cmap = plt.cm.Set2
    core_labels_seen = {}
    for cs, ce, lbl in labelled:
        if lbl is None:
            color = "#CCCCCC"
        else:
            idx = int(lbl[1:]) if lbl[1:].isdigit() else 0
            color = cmap(idx % 8)
        ax.add_patch(plt.Rectangle(
            (cs, core_y - core_h / 2), ce - cs, core_h,
            facecolor=color, edgecolor="#333333", linewidth=0.6, zorder=3))
        if lbl:
            mid = (cs + ce) / 2
            ax.text(mid, core_y, lbl, ha="center", va="center",
                    fontsize=6, fontweight="bold", zorder=4)

    ax.set_xlim(plot_start - (plot_end - plot_start) * 0.02,
                plot_end + (plot_end - plot_start) * 0.02)
    ax.set_ylim(-0.05, 1.0)
    ax.set_yticks([core_y, exon_y])
    ax.set_yticklabels(["Cores", "Exons"], fontsize=9)
    ax.set_xlabel("Genomic position (5\u2032 \u2192 3\u2032)" if is_minus else "Genomic position", fontsize=9)
    ax.set_title(f"{gene_symbol} — transcript structure with conserved cores",
                 fontweight="bold", fontsize=11)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    safe_name = gene_symbol.replace("-", "_")
    fig.savefig(FIGURES_DIR / f"{safe_name}_TRACK.pdf",
               format="pdf", bbox_inches="tight")
    plt.show()
    print(f"  Saved: {FIGURES_DIR / f'{safe_name}_TRACK.pdf'}")


for symbol, gid in TRACK_GENES:
    plot_transcript_cores(symbol, gid, bed, ref_data,
                          best_per_species)


In [ ]:
# Summary: case study genes — cores detected per species
genes_with_data = {sym: ensg for sym, ensg in CASE_STUDY_GENES.items()
                   if all_islands["gene_id_bare"].eq(ensg).any()
                   and best_per_species["gene_id"].isin(
                       all_islands.loc[all_islands["gene_id_bare"] == ensg, "gene_id"].unique()
                   ).any()}

n_genes = len(genes_with_data)
bar_width = 0.8 / max(n_genes, 1)
x = np.arange(len(PHYLO_ORDER))

fig, ax = plt.subplots(figsize=(14, 5))
for i, (symbol, ensg) in enumerate(genes_with_data.items()):
    gids = all_islands.loc[all_islands["gene_id_bare"] == ensg, "gene_id"].unique()
    sub = best_per_species[best_per_species["gene_id"].isin(gids)]
    n_cores_per_sp = sub.groupby("species")["core_id"].nunique().reindex(PHYLO_ORDER, fill_value=0)
    ax.bar(x + i * bar_width, n_cores_per_sp.values, bar_width, label=symbol)

ax.set_xticks(x + bar_width * (n_genes - 1) / 2)
ax.set_xticklabels([f"{SPECIES_META[s]['name']}\n({SPECIES_META[s]['div_mya']} My)"
                     for s in PHYLO_ORDER], fontsize=7.5)
ax.set_ylabel("Conserved cores detected")
ax.set_title("Case study lncRNAs: core detection across mammalian phylogeny")
ax.legend(fontsize=8, ncol=3, loc="upper right")
plt.tight_layout()
plt.show()

# Also note genes with no data
no_data = set(CASE_STUDY_GENES) - set(genes_with_data)
if no_data:
    print(f"\nGenes with no island data: {', '.join(sorted(no_data))}")

### 6.2 lncRNA-wide conservation landscape

Scatter of all lncRNA genes: how many species detect at least one core vs. median MMD.
Plus fraction-of-lncRNAs statistics at various conservation thresholds.

In [ ]:
# Build per-lncRNA-gene summary from conservation_with_mmd
lnc_cores = conservation_with_mmd[conservation_with_mmd["biotype"] == "lncRNA"].copy()
lnc_cores["gene_id_bare"] = (
    lnc_cores["gene_id"].str.replace("^U_", "", regex=True).str.split(".").str[0]
)
print(f"lncRNA conserved cores: {len(lnc_cores):,}")

lnc_genes = (
    lnc_cores.groupby("gene_id_bare")
    .agg(
        n_cores=("core_id", "count"),
        max_species=("n_species", "max"),
        mean_n_species=("n_species", "mean"),
        median_mmd=("median_mmd", "median"),
        mean_mmd=("mean_mmd", "mean"),
        gene_name=("gene_name", "first"),
    )
)
print(f"lncRNA genes with ≥1 core: {len(lnc_genes):,}")
lnc_genes.head()

In [ ]:
# Scatter: max #species with match vs median MMD per lncRNA gene
fig, ax = plt.subplots(figsize=(10, 7))

sc = ax.scatter(
    lnc_genes["max_species"], lnc_genes["median_mmd"],
    s=np.clip(lnc_genes["n_cores"] * 4, 4, 80),
    alpha=0.25, c="steelblue", edgecolors="none"
)

# Highlight case study genes
for symbol, ensg in CASE_STUDY_GENES.items():
    if ensg not in lnc_genes.index:
        continue
    r = lnc_genes.loc[ensg]
    ax.scatter(r["max_species"], r["median_mmd"],
               s=120, c="crimson", edgecolors="black", linewidths=0.8, zorder=5)
    ax.annotate(symbol, (r["max_species"], r["median_mmd"]),
                fontsize=8, fontweight="bold", ha="left", va="bottom",
                xytext=(5, 5), textcoords="offset points",
                bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))

ax.set_xlabel("Max number of species with ≥1 core match")
ax.set_ylabel("Median MMD (across species)")
ax.set_title("lncRNA conservation landscape (point size ∝ #cores)")
ax.set_xticks(range(1, 12))

# Median line per #species bin
medians = lnc_genes.groupby("max_species")["median_mmd"].median()
ax.plot(medians.index, medians.values, "ko-", ms=5, lw=1.5, zorder=4, label="Median trend")
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Fraction of lncRNAs meeting conservation criteria
total_lnc = len(lnc_genes)

print(f"Total lncRNA genes with ≥1 detected core: {total_lnc:,}\n")
print("=" * 65)
print(f"{'Criterion':<45s} {'Count':>7s} {'Fraction':>8s}")
print("=" * 65)

for k in range(1, 12):
    n = (lnc_genes["max_species"] >= k).sum()
    print(f"  ≥1 match in ≥{k:>2d} species                       {n:>7,d}  {n/total_lnc:>7.1%}")

print("-" * 65)

for nc in [2, 3, 5]:
    n = (lnc_genes["n_cores"] >= nc).sum()
    print(f"  ≥{nc} conserved cores                              {n:>7,d}  {n/total_lnc:>7.1%}")

print("-" * 65)

for thr in [0.05, 0.10, 0.15, 0.20]:
    n = (lnc_genes["mean_mmd"] < thr).sum()
    print(f"  Mean MMD < {thr:.2f}                                {n:>7,d}  {n/total_lnc:>7.1%}")

print("=" * 65)

# Combined: broadly conserved AND low MMD
for k, thr in [(8, 0.15), (10, 0.15), (11, 0.15), (8, 0.10), (10, 0.10)]:
    mask = (lnc_genes["max_species"] >= k) & (lnc_genes["mean_mmd"] < thr)
    n = mask.sum()
    print(f"  ≥{k} species AND mean MMD < {thr:.2f}               {n:>7,d}  {n/total_lnc:>7.1%}")

print("=" * 65)

## 7. Phylogenetic decay analysis

For islands found in many species, track how MMD changes with divergence time.

In [ ]:
# Aggregate: per divergence-time bin, what's the median MMD?
div_mmd = (
    best_per_species
    .groupby(["div_mya", "species"])
    .agg(median_mmd=("diag_mmd", "median"),
         mean_mmd=("diag_mmd", "mean"),
         q25_mmd=("diag_mmd", lambda x: x.quantile(0.25)),
         q75_mmd=("diag_mmd", lambda x: x.quantile(0.75)),
         n=("diag_mmd", "count"))
    .reset_index()
)

fig, ax = plt.subplots(figsize=(10, 6))
for _, row in div_mmd.iterrows():
    sp = row["species"]
    ax.errorbar(row["div_mya"], row["median_mmd"],
                yerr=[[row["median_mmd"] - row["q25_mmd"]], [row["q75_mmd"] - row["median_mmd"]]],
                fmt="o", ms=8, capsize=4, color="steelblue")
    ax.annotate(SPECIES_META[sp]["name"], (row["div_mya"], row["median_mmd"]),
                fontsize=7, ha="left", va="bottom", xytext=(4, 4), textcoords="offset points")

ax.set_xlabel("Divergence time from human (Mya)")
ax.set_ylabel("Median MMD (IQR bars)")
ax.set_title("Phylogenetic decay of island alignment quality")
plt.tight_layout()
plt.show()

In [ ]:
# Per-core decay: for highly-conserved cores, plot individual MMD trajectories
ultra_conserved = conservation_with_mmd[
    (conservation_with_mmd["n_species"] == len(PHYLO_ORDER)) &
    (conservation_with_mmd["mean_mmd"] < 0.10)
]["core_id"].values
print(f"Ultra-conserved cores (all {len(PHYLO_ORDER)} species, mean MMD < 0.10): {len(ultra_conserved):,}")

if len(ultra_conserved) > 0:
    sample_keys = ultra_conserved[:min(200, len(ultra_conserved))]
    sub = best_per_species[best_per_species["core_id"].isin(sample_keys)]

    fig, ax = plt.subplots(figsize=(10, 6))
    for key in sample_keys:
        ksub = sub[sub["core_id"] == key].sort_values("div_mya")
        ax.plot(ksub["div_mya"], ksub["diag_mmd"], alpha=0.15, color="steelblue", lw=0.7)

    # Add median trend
    trend = sub.groupby("div_mya")["diag_mmd"].median()
    ax.plot(trend.index, trend.values, "ko-", ms=6, lw=2.5, label="Median", zorder=5)

    ax.set_xlabel("Divergence time from human (Mya)")
    ax.set_ylabel("Diagonal MMD")
    ax.set_title(f"MMD trajectories for ultra-conserved islands (n={len(sample_keys)})")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No ultra-conserved islands with these criteria. Try relaxing thresholds.")

### 7.1 Decay patterns classification

Classify each island's phylogenetic behaviour:
- **Flat conserved**: low MMD in all species (< 0.10 everywhere)
- **Gradual decay**: positive correlation of MMD with divergence time
- **Sharp drop-off**: low in close species, high/missing in distant
- **Sporadic**: no clear pattern

In [ ]:
from scipy.stats import spearmanr

def classify_decay(group):
    """Classify an island's phylogenetic decay pattern."""
    n_sp = group["species"].nunique()
    if n_sp < 4:
        return "too_few_species"
    mmds = group.sort_values("div_mya")["diag_mmd"].values
    divs = group.sort_values("div_mya")["div_mya"].values
    
    if np.all(mmds < 0.10):
        return "flat_conserved"
    
    rho, pval = spearmanr(divs, mmds)
    if pval < 0.05 and rho > 0.5:
        return "gradual_decay"
    
    close_mmd = mmds[divs <= 30].mean() if (divs <= 30).any() else np.nan
    far_mmd = mmds[divs >= 90].mean() if (divs >= 90).any() else np.nan
    if not np.isnan(close_mmd) and not np.isnan(far_mmd) and (far_mmd - close_mmd) > 0.08:
        return "sharp_dropoff"
    
    return "sporadic"

# Apply to islands with enough data
qualifying_keys = conservation[conservation >= 4].index
qualifying_data = best_per_species[best_per_species["core_id"].isin(qualifying_keys)]

decay_classes = (
    qualifying_data
    .groupby("core_id")
    .apply(classify_decay)
    .rename("decay_class")
)

print("Decay pattern classification:")
print(decay_classes.value_counts())

In [ ]:
decay_with_meta = conservation_with_mmd.set_index("core_id").join(decay_classes)
decay_with_meta = decay_with_meta[decay_with_meta["decay_class"].notna()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
class_counts = decay_with_meta["decay_class"].value_counts()
colors = {"flat_conserved": "#2ca02c", "gradual_decay": "#ff7f0e",
          "sharp_dropoff": "#d62728", "sporadic": "#7f7f7f", "too_few_species": "#c7c7c7"}
ax.bar(range(len(class_counts)), class_counts.values,
       color=[colors.get(c, "gray") for c in class_counts.index])
ax.set_xticks(range(len(class_counts)))
ax.set_xticklabels(class_counts.index, rotation=30, ha="right")
ax.set_ylabel("Number of islands")
ax.set_title("Decay pattern classification")

ax = axes[1]
for cls in ["flat_conserved", "gradual_decay", "sharp_dropoff"]:
    csub = decay_with_meta[decay_with_meta["decay_class"] == cls]
    if len(csub) == 0:
        continue
    ax.scatter(csub["n_species"], csub["mean_mmd"], alpha=0.3, s=10,
               color=colors[cls], label=cls)
ax.set_xlabel("Number of species")
ax.set_ylabel("Mean MMD")
ax.set_title("Decay class vs. conservation breadth")
ax.legend(fontsize=8, markerscale=3)

plt.tight_layout()
plt.show()

## 8. Top conserved islands

Islands found in all 10 species with the lowest mean MMD.

In [ ]:
top_conserved = (
    conservation_with_mmd[conservation_with_mmd["n_species"] == len(PHYLO_ORDER)]
    .sort_values("mean_mmd")
    .head(30)
)

display_cols = ["core_id", "gene_name", "biotype", "n_species", "mean_mmd",
                "median_mmd", "min_mmd", "max_mmd", "mean_ref_len"]
display_cols = [c for c in display_cols if c in top_conserved.columns]
print(f"Top 30 islands found in all {len(PHYLO_ORDER)} species (sorted by mean MMD):")
top_conserved[display_cols]

In [ ]:
# Mini heatmap for top 30 (species=Y, cores=X)
if len(top_conserved) > 0:
    top_keys = top_conserved["core_id"].values
    mini_heatmap = (
        best_per_species[best_per_species["core_id"].isin(top_keys)]
        .pivot_table(index="species", columns="core_id", values="diag_mmd", aggfunc="min")
        .reindex(index=PHYLO_ORDER, columns=top_keys)
    )

    key_to_name = top_conserved.set_index("core_id")["gene_name"].to_dict()
    col_labels = []
    for k in top_keys:
        gn = key_to_name.get(k)
        core_num = k.rsplit("_C", 1)[-1]
        label = f"{gn} (C{core_num})" if pd.notna(gn) else k
        col_labels.append(label)

    species_labels = [SPECIES_META[s]["name"] for s in PHYLO_ORDER]

    fig, ax = plt.subplots(figsize=(max(8, len(top_keys) * 0.55 + 2), 6))
    im = ax.imshow(mini_heatmap.values, aspect="auto", cmap="RdYlGn_r", vmin=0, vmax=0.25)
    ax.set_xticks(range(len(top_keys)))
    ax.set_xticklabels(col_labels, rotation=60, ha="right", fontsize=7)
    ax.set_yticks(range(len(PHYLO_ORDER)))
    ax.set_yticklabels(species_labels, fontsize=8)
    ax.set_title("Top 30 most conserved islands – MMD heatmap")
    plt.colorbar(im, ax=ax, label="MMD", shrink=0.6)
    plt.tight_layout()
    plt.show()


## 9. "Found everywhere except opossum" pattern

Islands detected in all 9 placental mammals but missing or high-MMD in opossum.

In [ ]:
placental_species = [s for s in PHYLO_ORDER if s != "monDom5"]

placental_sub = best_per_species[best_per_species["species"].isin(placental_species)]
opossum_sub = best_per_species[best_per_species["species"] == "monDom5"]

placental_conservation = placental_sub.groupby("core_id")["species"].nunique()
opossum_found = set(opossum_sub["core_id"].unique())

# Cores in all 10 placentals
all_placentals = set(placental_conservation[placental_conservation == len(placental_species)].index)
print(f"Cores found in all {len(placental_species)} placental species: {len(all_placentals):,}")

# Of those, missing in opossum
missing_in_opossum = all_placentals - opossum_found
print(f"  ... and MISSING in opossum: {len(missing_in_opossum):,}")

# Found in opossum but with high MMD
found_in_opossum = all_placentals & opossum_found
opossum_mmds = opossum_sub[opossum_sub["core_id"].isin(found_in_opossum)]
high_mmd_opossum = set(opossum_mmds[opossum_mmds["diag_mmd"] > 0.2]["core_id"])
print(f"  ... found in opossum but MMD > 0.2: {len(high_mmd_opossum):,}")

placental_only = missing_in_opossum | high_mmd_opossum
print(f"\nTotal placental-enriched cores: {len(placental_only):,}")

In [ ]:
# MMD in placentals for these cores
plac_only_mmd = (
    placental_sub[placental_sub["core_id"].isin(placental_only)]
    .groupby("core_id")["diag_mmd"].mean()
    .sort_values()
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(plac_only_mmd.values, bins=50, color="darkcyan", edgecolor="white")
ax.set_xlabel("Mean placental MMD")
ax.set_ylabel("Number of islands")
ax.set_title(f"Placental-enriched islands (n={len(plac_only_mmd):,}): MMD distribution")
ax.axvline(plac_only_mmd.median(), ls="--", color="red", label=f"Median={plac_only_mmd.median():.3f}")
ax.legend()
plt.tight_layout()
plt.show()

## 10. Detection rate heatmap: gene-level (fraction of cores detected per gene per species)

In [ ]:
# Total conserved cores per gene
total_cores_per_gene = all_islands.drop_duplicates("core_id")["gene_id"].value_counts()

# Detected cores per gene per species
detected = (
    best_per_species
    .groupby(["gene_id", "species"])["core_id"]
    .nunique()
    .reset_index(name="n_detected")
)
detected["n_total"] = detected["gene_id"].map(total_cores_per_gene)
detected["frac_detected"] = detected["n_detected"] / detected["n_total"]

gene_detection = detected.pivot_table(
    index="gene_id", columns="species", values="frac_detected"
)[PHYLO_ORDER].fillna(0)

print(f"Gene detection matrix shape: {gene_detection.shape}")
print(f"\nMean fraction of cores detected per species:")
for sp in PHYLO_ORDER:
    print(f"  {SPECIES_META[sp]['name']:20s}: {gene_detection[sp].mean():.3f}")

In [ ]:
# Summary heatmap: mean detection rate per biotype x species
detected_with_bt = detected.copy()
detected_with_bt["biotype"] = detected_with_bt["gene_id"].map(biotype_map)

top_bts = detected_with_bt["biotype"].value_counts().head(8).index
bt_detection = (
    detected_with_bt[detected_with_bt["biotype"].isin(top_bts)]
    .groupby(["biotype", "species"])["frac_detected"]
    .mean()
    .unstack("species")
    [PHYLO_ORDER]
)

fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(bt_detection.values, aspect="auto", cmap="YlGnBu", vmin=0, vmax=1)
ax.set_xticks(range(len(PHYLO_ORDER)))
ax.set_xticklabels([SPECIES_META[s]["name"] for s in PHYLO_ORDER], rotation=45, ha="right")
ax.set_yticks(range(len(bt_detection)))
ax.set_yticklabels(bt_detection.index)
ax.set_title("Mean core detection rate: biotype × species")
plt.colorbar(im, ax=ax, label="Fraction of cores detected", shrink=0.7)
plt.tight_layout()
plt.show()

## 11. MMD vs aligned length (scatter, coloured by divergence)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

sample = best_per_species.sample(n=min(50000, len(best_per_species)), random_state=42)
sc = ax.scatter(
    sample["ref_len"], sample["diag_mmd"],
    c=sample["div_mya"], cmap="plasma", s=3, alpha=0.3,
    vmin=0, vmax=200
)
ax.set_xlabel("Aligned reference length (bp)")
ax.set_ylabel("Diagonal MMD")
ax.set_title("MMD vs. aligned island length (coloured by divergence time)")
ax.set_xlim(0, sample["ref_len"].quantile(0.99))
ax.set_ylim(0, 0.5)
plt.colorbar(sc, ax=ax, label="Divergence time (Mya)", shrink=0.7)
plt.tight_layout()
plt.show()

## 12. Biotype-stratified MMD decay

In [ ]:
focus_biotypes = ["lncRNA", "protein_coding", "processed_pseudogene", "miRNA", "snRNA", "snoRNA"]
bt_sub = best_per_species[best_per_species["biotype"].isin(focus_biotypes)]

fig, axes = plt.subplots(2, 3, figsize=(16, 10), sharey=True)
for ax, bt in zip(axes.flat, focus_biotypes):
    bsub = bt_sub[bt_sub["biotype"] == bt]
    agg = bsub.groupby("species").agg(
        div_mya=("div_mya", "first"),
        median_mmd=("diag_mmd", "median"),
        q25=("diag_mmd", lambda x: x.quantile(0.25)),
        q75=("diag_mmd", lambda x: x.quantile(0.75)),
        n=("diag_mmd", "count")
    ).sort_values("div_mya")

    ax.errorbar(agg["div_mya"], agg["median_mmd"],
                yerr=[agg["median_mmd"] - agg["q25"], agg["q75"] - agg["median_mmd"]],
                fmt="o-", ms=6, capsize=3, color="steelblue")
    ax.set_title(bt, fontsize=11, fontweight="bold")
    ax.set_xlabel("Div. time (Mya)")
    if ax in axes[:, 0]:
        ax.set_ylabel("Median MMD (IQR)")

plt.suptitle("MMD phylogenetic decay by biotype", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 13. Summary statistics table

In [ ]:
summary = []
for bt in focus_biotypes:
    bsub = conservation_with_mmd[conservation_with_mmd["biotype"] == bt]
    summary.append({
        "biotype": bt,
        "total_islands": len(bsub),
        f"in_all_{len(PHYLO_ORDER)}": (bsub["n_species"] == len(PHYLO_ORDER)).sum(),
        "in_9+": (bsub["n_species"] >= 9).sum(),
        "in_8+": (bsub["n_species"] >= 8).sum(),
        f"frac_in_all_{len(PHYLO_ORDER)}": f"{(bsub['n_species'] == len(PHYLO_ORDER)).mean():.1%}" if len(bsub) else "N/A",
        "median_mean_mmd": f"{bsub['mean_mmd'].median():.4f}" if len(bsub) else "N/A",
        "median_ref_len": f"{bsub['mean_ref_len'].median():.0f}" if len(bsub) else "N/A",
    })

pd.DataFrame(summary)

## 14. Per-gene island count vs conservation

In [ ]:
gene_level = (
    conservation_with_mmd
    .groupby("gene_id")
    .agg(
        n_cores=("core_id", "count"),
        mean_conservation=("n_species", "mean"),
        max_conservation=("n_species", "max"),
        mean_mmd=("mean_mmd", "mean"),
        biotype=("biotype", "first"),
        gene_name=("gene_name", "first"),
    )
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(gene_level["n_cores"], gene_level["mean_conservation"],
           alpha=0.1, s=5, color="steelblue")
ax.set_xlabel("Number of conserved cores per gene")
ax.set_ylabel("Mean species conservation")
ax.set_title("Island count vs. conservation breadth")

ax = axes[1]
for bt in ["lncRNA", "protein_coding", "processed_pseudogene"]:
    sub = gene_level[gene_level["biotype"] == bt]
    ax.hist(sub["mean_conservation"], bins=11, range=(0.5, 11.5),
            alpha=0.5, label=bt, density=True)
ax.set_xlabel("Mean species conservation per gene")
ax.set_ylabel("Density")
ax.set_title("Gene-level conservation by biotype")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Top genes by mean conservation
print("Top 20 most broadly conserved genes (by mean core conservation):")
gene_level.sort_values("mean_conservation", ascending=False).head(20)[
    ["gene_name", "biotype", "n_cores", "mean_conservation", "max_conservation", "mean_mmd"]
]

## 15. lncRNA overlap with protein-coding genes

How many of the "conserved lncRNAs" are antisense to or overlapping protein-coding genes?
Classification:
- **Antisense**: lncRNA overlaps a coding gene on the opposite strand
- **Sense-overlapping**: lncRNA overlaps a coding gene on the same strand
- **Intergenic**: no genomic overlap with any coding gene

In [ ]:
full_bed_path = ANNOTATION_DIR / "hg38.primary_only.bed"
full_meta_path = ANNOTATION_DIR / "hg38.primary_only.transcript_metadata.tsv"
gene_type_path = ANNOTATION_DIR / "hg38_gene_names.txt"

tx_bed = pd.read_csv(
    full_bed_path, sep="\t", header=None,
    usecols=[0, 1, 2, 3, 5],
    names=["chrom", "start", "end", "tx_id", "strand"],
    dtype={"start": int, "end": int},
)
tx_meta = pd.read_csv(full_meta_path, sep="\t")
tx_bed = tx_bed.merge(tx_meta, left_on="tx_id", right_on="transcript_id", how="left")

gene_types = pd.read_csv(gene_type_path, sep="\t", usecols=["Gene stable ID", "Gene type"])
gene_types = gene_types.drop_duplicates("Gene stable ID").set_index("Gene stable ID")["Gene type"]

tx_bed["gene_id_bare"] = tx_bed["gene_id"].str.split(".").str[0]
tx_bed["gene_biotype"] = tx_bed["gene_id_bare"].map(gene_types)
tx_bed["gene_biotype"] = tx_bed["gene_biotype"].fillna(tx_bed["transcript_biotype"])

gene_loci = (
    tx_bed.groupby("gene_id_bare")
    .agg(
        chrom=("chrom", "first"),
        start=("start", "min"),
        end=("end", "max"),
        strand=("strand", "first"),
        biotype=("gene_biotype", "first"),
    )
    .reset_index()
)

coding_loci = gene_loci[gene_loci["biotype"] == "protein_coding"].copy()
lnc_loci = gene_loci[gene_loci["biotype"] == "lncRNA"].copy()
print(f"Gene loci built: {len(gene_loci):,} total")
print(f"  protein_coding: {len(coding_loci):,}")
print(f"  lncRNA:         {len(lnc_loci):,}")

In [ ]:
overlap_records = []

for chrom in sorted(lnc_loci["chrom"].unique()):
    lnc_ch = lnc_loci[lnc_loci["chrom"] == chrom]
    cod_ch = coding_loci[coding_loci["chrom"] == chrom]
    if cod_ch.empty:
        for gid in lnc_ch["gene_id_bare"]:
            overlap_records.append((gid, "intergenic", 0, ""))
        continue

    ls = lnc_ch["start"].values[:, None]
    le = lnc_ch["end"].values[:, None]
    cs = cod_ch["start"].values[None, :]
    ce = cod_ch["end"].values[None, :]

    # BED half-open overlap: ls < ce AND cs < le
    ov_matrix = (ls < ce) & (cs < le)

    lnc_strands = lnc_ch["strand"].values
    cod_strands = cod_ch["strand"].values
    cod_genes = cod_ch["gene_id_bare"].values

    for i, (gid, l_str) in enumerate(zip(lnc_ch["gene_id_bare"].values, lnc_strands)):
        hits = np.where(ov_matrix[i])[0]
        if len(hits) == 0:
            overlap_records.append((gid, "intergenic", 0, ""))
        else:
            hit_strands = cod_strands[hits]
            has_anti = np.any(hit_strands != l_str)
            has_sense = np.any(hit_strands == l_str)
            if has_anti and has_sense:
                cat = "antisense+sense"
            elif has_anti:
                cat = "antisense"
            else:
                cat = "sense_overlapping"
            partners = ",".join(cod_genes[hits][:5])
            overlap_records.append((gid, cat, len(hits), partners))

overlap_df = pd.DataFrame(
    overlap_records,
    columns=["gene_id_bare", "overlap_category", "n_coding_overlaps", "coding_partners"],
)

PROXIMITY_BP = 5000

intergenic_mask = overlap_df["overlap_category"] == "intergenic"
intergenic_genes = set(overlap_df.loc[intergenic_mask, "gene_id_bare"])
lnc_ig = lnc_loci[lnc_loci["gene_id_bare"].isin(intergenic_genes)]

near_coding = set()
for chrom in sorted(lnc_ig["chrom"].unique()):
    lig = lnc_ig[lnc_ig["chrom"] == chrom]
    cod = coding_loci[coding_loci["chrom"] == chrom]
    if cod.empty:
        continue
    ls = lig["start"].values[:, None]
    le = lig["end"].values[:, None]
    cs = cod["start"].values[None, :] - PROXIMITY_BP
    ce = cod["end"].values[None, :] + PROXIMITY_BP
    prox_matrix = (ls < ce) & (cs < le)
    hit_rows = np.where(prox_matrix.any(axis=1))[0]
    near_coding.update(lig.iloc[hit_rows]["gene_id_bare"].values)

overlap_df.loc[
    intergenic_mask & overlap_df["gene_id_bare"].isin(near_coding),
    "overlap_category",
] = "near_coding_5kb"

print("ALL lncRNAs in reference annotation — overlap with protein-coding genes:")
print(overlap_df["overlap_category"].value_counts())
print(f"\nTotal: {len(overlap_df):,}")
frac_strict = (overlap_df["overlap_category"] == "intergenic").mean()
frac_near = (overlap_df["overlap_category"] == "near_coding_5kb").mean()
print(f"Strictly intergenic (>5kb from coding): {frac_strict:.1%}")
print(f"Near-coding (within ±5kb, no overlap):  {frac_near:.1%}")

### 15.1 Overlap classification for conserved lncRNAs

Join the overlap classification back to the lncRNA conservation data to see how many
of our "conserved lncRNAs" are truly intergenic vs coding-adjacent.

In [ ]:
lnc_genes_ov = lnc_genes.join(
    overlap_df.set_index("gene_id_bare")[["overlap_category", "n_coding_overlaps", "coding_partners"]],
    how="left",
)
lnc_genes_ov["overlap_category"] = lnc_genes_ov["overlap_category"].fillna("unknown")

print("Conserved lncRNAs (genes with ≥1 detected core) — overlap with coding genes:")
cat_counts = lnc_genes_ov["overlap_category"].value_counts()
print(cat_counts)
print(f"\nTotal conserved lncRNA genes: {len(lnc_genes_ov):,}")
print(f"Strictly intergenic (>5kb): {(lnc_genes_ov['overlap_category'] == 'intergenic').sum():,}"
      f" ({(lnc_genes_ov['overlap_category'] == 'intergenic').mean():.1%})")
print(f"Near-coding (±5kb):         {(lnc_genes_ov['overlap_category'] == 'near_coding_5kb').sum():,}"
      f" ({(lnc_genes_ov['overlap_category'] == 'near_coding_5kb').mean():.1%})")

print("\n--- Breakdown by conservation breadth (max #species with a core) ---")
for min_sp in [1, 3, 5, 7, len(PHYLO_ORDER)]:
    sub = lnc_genes_ov[lnc_genes_ov["max_species"] >= min_sp]
    if sub.empty:
        continue
    cats = sub["overlap_category"].value_counts()
    n_inter = cats.get("intergenic", 0)
    n_near = cats.get("near_coding_5kb", 0)
    n_anti = cats.get("antisense", 0) + cats.get("antisense+sense", 0)
    n_sense = cats.get("sense_overlapping", 0) + cats.get("antisense+sense", 0)
    print(f"\n  ≥{min_sp} species (n={len(sub):,}):")
    print(f"    intergenic (>5kb):{n_inter:5d}  ({n_inter/len(sub):.1%})")
    print(f"    near-coding ±5kb: {n_near:5d}  ({n_near/len(sub):.1%})")
    print(f"    antisense:        {n_anti:5d}  ({n_anti/len(sub):.1%})")
    print(f"    sense-overlapping:{n_sense:5d}  ({n_sense/len(sub):.1%})")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

cat_order = ["intergenic", "near_coding_5kb", "antisense", "sense_overlapping", "antisense+sense"]
cat_colors = {"intergenic": "#2196F3", "near_coding_5kb": "#4CAF50",
              "antisense": "#FF9800", "sense_overlapping": "#F44336",
              "antisense+sense": "#9C27B0", "unknown": "#999"}

# Panel 1: pie chart of all conserved lncRNAs
ax = axes[0]
cats = lnc_genes_ov["overlap_category"].value_counts()
labels = [c for c in cat_order if c in cats.index]
vals = [cats[c] for c in labels]
colors = [cat_colors[c] for c in labels]
ax.pie(vals, labels=[f"{l}\n({v:,})" for l, v in zip(labels, vals)],
       colors=colors, autopct="%.1f%%", startangle=90)
ax.set_title("All conserved lncRNAs", fontweight="bold")

# Panel 2: stacked bar by conservation breadth
ax = axes[1]
thresholds = [1, 2, 3, 5, 7, len(PHYLO_ORDER)]
bar_data = {c: [] for c in cat_order}
xticks = []
for t in thresholds:
    sub = lnc_genes_ov[lnc_genes_ov["max_species"] >= t]
    total = max(len(sub), 1)
    xticks.append(f"≥{t}")
    for c in cat_order:
        bar_data[c].append((sub["overlap_category"] == c).sum() / total * 100)

x = np.arange(len(thresholds))
bottom = np.zeros(len(thresholds))
for c in cat_order:
    vals = np.array(bar_data[c])
    ax.bar(x, vals, bottom=bottom, color=cat_colors[c], label=c, width=0.7)
    bottom += vals
ax.set_xticks(x)
ax.set_xticklabels(xticks)
ax.set_xlabel("Min. #species with ≥1 core")
ax.set_ylabel("Fraction (%)")
ax.set_title("Overlap by conservation breadth", fontweight="bold")
ax.legend(fontsize=7, loc="upper right")

# Panel 3: scatter of conservation vs MMD, colored by overlap category
ax = axes[2]
for cat in cat_order:
    sub = lnc_genes_ov[lnc_genes_ov["overlap_category"] == cat]
    ax.scatter(sub["max_species"], sub["median_mmd"],
               alpha=0.3, s=8, color=cat_colors[cat], label=cat, rasterized=True)
ax.set_xlabel("Max #species with ≥1 core")
ax.set_ylabel("Median MMD")
ax.set_title("Conservation landscape by overlap", fontweight="bold")
ax.legend(fontsize=7, markerscale=3)

plt.tight_layout()
plt.show()

### 15.2 Truly intergenic conserved lncRNAs

After removing coding-overlapping lncRNAs, what remains?

In [ ]:
intergenic = lnc_genes_ov[lnc_genes_ov["overlap_category"] == "intergenic"].copy()
intergenic_broad = intergenic[intergenic["max_species"] >= 5].sort_values("median_mmd")

print(f"Intergenic lncRNAs with cores in ≥5 species: {len(intergenic_broad):,}")
print(f"  (vs {(lnc_genes_ov['max_species'] >= 5).sum():,} total lncRNAs at that threshold)\n")
print("Top 30 by lowest median MMD:")
display_cols = ["gene_name", "n_cores", "max_species", "median_mmd", "mean_mmd"]
intergenic_broad.head(30)[display_cols]

In [ ]:
print("Case study genes — overlap classification:")
print("=" * 70)
for sym, ensg in CASE_STUDY_GENES.items():
    row = overlap_df[overlap_df["gene_id_bare"] == ensg]
    if row.empty:
        cat, partners = "not in annotation", ""
    else:
        cat = row.iloc[0]["overlap_category"]
        partners = row.iloc[0]["coding_partners"]
    tag = f"  → overlaps: {partners}" if partners else ""
    print(f"  {sym:15s} ({ensg})  {cat}{tag}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: count by conservation threshold — 3 buckets
ax = axes[0]
thresholds = list(range(1, len(PHYLO_ORDER) + 1))
n_ig, n_near, n_ov = [], [], []
for t in thresholds:
    sub = lnc_genes_ov[lnc_genes_ov["max_species"] >= t]
    n_ig.append((sub["overlap_category"] == "intergenic").sum())
    n_near.append((sub["overlap_category"] == "near_coding_5kb").sum())
    n_ov.append(sub["overlap_category"].isin(
        ["antisense", "sense_overlapping", "antisense+sense"]).sum())
ax.plot(thresholds, n_ig, "o-", color="#2196F3", label="intergenic (>5kb)", linewidth=2)
ax.plot(thresholds, n_near, "D-", color="#4CAF50", label="near-coding (±5kb)", linewidth=2)
ax.plot(thresholds, n_ov, "s-", color="#FF9800", label="overlapping coding", linewidth=2)
ax.set_xlabel("Min. #species with ≥1 core")
ax.set_ylabel("Number of lncRNA genes")
ax.set_title("lncRNA proximity to coding genes", fontweight="bold")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Panel 2: MMD distributions — 3 buckets
ax = axes[1]
ig = lnc_genes_ov[lnc_genes_ov["overlap_category"] == "intergenic"]["median_mmd"].dropna()
nr = lnc_genes_ov[lnc_genes_ov["overlap_category"] == "near_coding_5kb"]["median_mmd"].dropna()
ov = lnc_genes_ov[lnc_genes_ov["overlap_category"].isin(
    ["antisense", "sense_overlapping", "antisense+sense"])]["median_mmd"].dropna()
ax.hist(ig, bins=50, alpha=0.55, color="#2196F3", label=f"intergenic >5kb (n={len(ig):,})", density=True)
ax.hist(nr, bins=50, alpha=0.55, color="#4CAF50", label=f"near-coding ±5kb (n={len(nr):,})", density=True)
ax.hist(ov, bins=50, alpha=0.55, color="#FF9800", label=f"overlapping (n={len(ov):,})", density=True)
ax.set_xlabel("Median MMD")
ax.set_ylabel("Density")
ax.set_title("MMD distribution by coding proximity", fontweight="bold")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"Intergenic (>5kb):   median MMD = {ig.median():.4f}  (n={len(ig):,})")
print(f"Near-coding (±5kb):  median MMD = {nr.median():.4f}  (n={len(nr):,})")
print(f"Overlapping coding:  median MMD = {ov.median():.4f}  (n={len(ov):,})")

### 15.3 Top conserved intergenic lncRNAs

Stringent filter: at least one core found in ≥ (N-1) species, mean aligned length > 120 bp,
low MMD. How many intergenic lncRNAs qualify?

In [ ]:
MIN_SPECIES = len(PHYLO_ORDER) - 1
MIN_LEN = 120

lnc_cwm = conservation_with_mmd[conservation_with_mmd["biotype"] == "lncRNA"].copy()
lnc_cwm["gene_id_bare"] = (
    lnc_cwm["gene_id"].str.replace("^U_", "", regex=True).str.split(".").str[0]
)

highly_conserved_cores = lnc_cwm[
    (lnc_cwm["n_species"] >= MIN_SPECIES) &
    (lnc_cwm["mean_ref_len"] >= MIN_LEN)
].copy()
print(f"lncRNA cores in ≥{MIN_SPECIES} species with mean length ≥{MIN_LEN} bp: "
      f"{len(highly_conserved_cores):,}")

hc_genes = (
    highly_conserved_cores.groupby("gene_id_bare")
    .agg(
        gene_name=("gene_name", "first"),
        n_qualifying_cores=("core_id", "count"),
        best_mean_mmd=("mean_mmd", "min"),
        best_median_mmd=("median_mmd", "min"),
        max_mean_len=("mean_ref_len", "max"),
        max_species=("n_species", "max"),
    )
    .sort_values("best_mean_mmd")
)

hc_genes = hc_genes.join(
    overlap_df.set_index("gene_id_bare")[["overlap_category", "coding_partners"]],
    how="left",
)
hc_genes["overlap_category"] = hc_genes["overlap_category"].fillna("unknown")

n_total = len(hc_genes)
n_intergenic = (hc_genes["overlap_category"] == "intergenic").sum()
n_antisense = hc_genes["overlap_category"].isin(["antisense", "antisense+sense"]).sum()
n_sense = hc_genes["overlap_category"].isin(["sense_overlapping", "antisense+sense"]).sum()

print(f"\nUnique lncRNA GENES with ≥1 qualifying core: {n_total}")
print(f"  intergenic:        {n_intergenic}  ({n_intergenic/n_total:.1%})")
print(f"  antisense:         {n_antisense}  ({n_antisense/n_total:.1%})")
print(f"  sense-overlapping: {n_sense}  ({n_sense/n_total:.1%})")

print(f"\n{'='*90}")
print(f"TOP INTERGENIC lncRNAs — core in ≥{MIN_SPECIES} species, ≥{MIN_LEN} bp, sorted by MMD")
print(f"{'='*90}")
ig_top = hc_genes[hc_genes["overlap_category"] == "intergenic"]
ig_top[["gene_name", "n_qualifying_cores", "max_species",
        "best_mean_mmd", "best_median_mmd", "max_mean_len"]]

In [ ]:
print(f"{'='*90}")
print(f"ALL qualifying genes (any overlap category), sorted by MMD")
print(f"{'='*90}")
hc_genes[["gene_name", "overlap_category", "coding_partners",
          "n_qualifying_cores", "max_species", "best_mean_mmd",
          "best_median_mmd", "max_mean_len"]]

In [ ]:
ov_lookup = overlap_df.set_index("gene_id_bare")["overlap_category"]

print("Breakdown at different MMD thresholds:")
print(f"(cores in ≥{MIN_SPECIES} species, ≥{MIN_LEN} bp)\n")
print(f"{'MMD thresh':>12s}  {'total':>6s}  {'intergenic':>10s}  {'near ±5kb':>10s}  {'overlapping':>11s}")
print("-" * 60)

for mmd_thresh in [0.05, 0.10, 0.15, 0.20, 0.25, 1.0]:
    sub = lnc_cwm[
        (lnc_cwm["n_species"] >= MIN_SPECIES) &
        (lnc_cwm["mean_ref_len"] >= MIN_LEN) &
        (lnc_cwm["mean_mmd"] <= mmd_thresh)
    ]
    genes = sub["gene_id_bare"].unique()
    cats = [ov_lookup.get(g, "unknown") for g in genes]
    n_ig = sum(1 for c in cats if c == "intergenic")
    n_near = sum(1 for c in cats if c == "near_coding_5kb")
    n_ov = sum(1 for c in cats if c in ("antisense", "sense_overlapping", "antisense+sense"))
    label = "no filter" if mmd_thresh >= 1.0 else f"≤{mmd_thresh:.2f}"
    print(f"  {label:>10s}  {len(genes):6d}  {n_ig:10d}  {n_near:10d}  {n_ov:11d}")

In [ ]:
MMD_THRESH = 0.10

_filtered = lnc_cwm[
    (lnc_cwm["n_species"] >= MIN_SPECIES) &
    (lnc_cwm["mean_ref_len"] >= MIN_LEN) &
    (lnc_cwm["mean_mmd"] <= MMD_THRESH)
].copy()

_genes = (
    _filtered.groupby("gene_id_bare")
    .agg(
        gene_name=("gene_name", "first"),
        n_cores=("core_id", "count"),
        best_mean_mmd=("mean_mmd", "min"),
        best_median_mmd=("median_mmd", "min"),
        max_len=("mean_ref_len", "max"),
        max_species=("n_species", "max"),
    )
    .sort_values("best_mean_mmd")
)
_genes = _genes.join(overlap_df.set_index("gene_id_bare")["overlap_category"], how="left")

def _print_gene_table(df, title):
    is_unnamed = df["gene_name"].str.match(r"^ENSG\d+$")
    n_unnamed = is_unnamed.sum()
    n_named = len(df) - n_unnamed
    print(f"\n{'='*105}")
    print(f"{title}")
    print(f"Total: {len(df)}  |  Named: {n_named}  |  Unnamed (ENSG only): {n_unnamed}")
    print(f"{'='*105}")
    print(f"{'#':>4s}  {'ENSG':17s} {'Gene name':20s} {'cores':>5s} {'sp':>3s}"
          f" {'mean_MMD':>9s} {'med_MMD':>8s} {'max_len':>8s}")
    print(f"{'-'*105}")
    for rank, (gid, row) in enumerate(df.iterrows(), 1):
        name = row["gene_name"] if not str(row["gene_name"]).startswith("ENSG") else "—"
        print(f"{rank:4d}  {gid:17s} {name:20s} {row['n_cores']:5d} {row['max_species']:3.0f}"
              f" {row['best_mean_mmd']:9.4f} {row['best_median_mmd']:8.4f} {row['max_len']:8.0f}")

ig_strict = _genes[_genes["overlap_category"] == "intergenic"]
ig_near = _genes[_genes["overlap_category"] == "near_coding_5kb"]

_print_gene_table(
    ig_strict,
    f"INTERGENIC (>5kb from coding) lncRNAs — ≥{MIN_SPECIES} sp, ≥{MIN_LEN} bp, MMD ≤{MMD_THRESH}",
)
_print_gene_table(
    ig_near,
    f"NEAR-CODING (±5kb) lncRNAs — ≥{MIN_SPECIES} sp, ≥{MIN_LEN} bp, MMD ≤{MMD_THRESH}",
)

## Preprint summary statistics

Numbers for the text: island detection rates, median islands per transcript,
cross-species match statistics, annotation overlap.

In [ ]:
import json as _json
from collections import Counter

REF_JSON_PATH = RESULTS_DIR / "hg38_vs_mm39" / "preprocessed_reference_data.json"
META_PATH = RESULTS_DIR / "hg38_vs_mm39" / "reference_union_transcripts_metadata.tsv"

with open(REF_JSON_PATH) as _f:
    _ref = _json.load(_f)

_meta = pd.read_csv(META_PATH, sep="\t")
_bt_map = _meta.set_index("transcript_id")["biotype"].to_dict()

MIN_ISL_LEN = 72  # same as pipeline IslandAlignmentConfig.min_island_len

# --- 1. Island detection per transcript ---
lnc_total = 0
lnc_with_islands = 0
lnc_island_counts = []
all_island_counts = []

for gid, data in _ref.items():
    bt = _bt_map.get(gid, "unknown")
    islands = [i for i in data.get("islands", []) if i["end"] - i["start"] >= MIN_ISL_LEN]
    n = len(islands)
    all_island_counts.append(n)
    is_lnc = "lnc" in bt.lower() or "linc" in bt.lower()
    if is_lnc:
        lnc_total += 1
        if n > 0:
            lnc_with_islands += 1
            lnc_island_counts.append(n)

lnc_frac = lnc_with_islands / lnc_total * 100
lnc_median = np.median(lnc_island_counts)
lnc_dist = Counter(lnc_island_counts)

# --- 2. Cross-species match statistics ---
n_total_cores = len(conservation)
n_multi = (conservation >= 2).sum()
n_half = (conservation >= 5).sum()
n_all = (conservation >= 10).sum()

total_matched_genes = all_islands["gene_id"].nunique()
genes_in_ref = len(_ref)
match_rate = total_matched_genes / genes_in_ref * 100

# Partial conservation: genes where only some cores match
cores_per_gene = conservation_with_mmd.groupby("gene_id").agg(
    n_cores=("n_species", "count"),
    max_sp=("n_species", "max"),
    mean_sp=("n_species", "mean"),
)

# --- 3. Annotation overlap (matched query islands vs existing annotations) ---
# For mm39: check how many matched query islands fall in regions that are NOT
# already annotated as ncRNA. We approximate this by checking if query islands
# from the lncRNA_islands.bed are present in the query_islands.bed (all detected)
# vs the full query annotation.
# NOTE: proper overlap requires bedtools intersect with Ensembl ncRNA annotation.
# Here we report what fraction of reference genes with matched islands are
# already annotated as lncRNA in the reference.

print("=" * 70)
print("PREPRINT SUMMARY STATISTICS")
print("=" * 70)

print(f"\n--- Island detection (reference, hg38) ---")
print(f"Total transcripts in reference:       {len(_ref):>8,}")
print(f"  of which lncRNAs:                   {lnc_total:>8,}")
print(f"lncRNAs with ≥1 island (≥{MIN_ISL_LEN}bp):   {lnc_with_islands:>8,}  ({lnc_frac:.1f}%)")
print(f"Median islands per lncRNA (w/ islands):{lnc_median:>7.0f}")
print(f"Distribution of island counts:")
for k in sorted(lnc_dist.keys())[:6]:
    print(f"  {k} island(s): {lnc_dist[k]:>5,}  ({lnc_dist[k]/lnc_with_islands:.1%})")
print(f"  >6:           {sum(v for k,v in lnc_dist.items() if k>6):>5,}")

print(f"\n--- Cross-species conservation ---")
print(f"Species compared:                     {len(PHYLO_ORDER):>8}")
print(f"Total conserved cores:                {n_total_cores:>8,}")
print(f"Cores in ≥2 species:                  {n_multi:>8,}  ({n_multi/n_total_cores:.1%})")
print(f"Cores in ≥5 species:                  {n_half:>8,}  ({n_half/n_total_cores:.1%})")
print(f"Cores in all {len(PHYLO_ORDER)} species:              {n_all:>8,}  ({n_all/n_total_cores:.1%})")
print(f"Genes with ≥1 cross-species match:    {total_matched_genes:>8,}  ({match_rate:.1f}% of ref)")

print(f"\n--- Match quality ---")
mean_mmd_all = all_islands["diag_mmd"].astype(float).mean()
median_mmd_all = all_islands["diag_mmd"].astype(float).median()
q25_mmd = all_islands["diag_mmd"].astype(float).quantile(0.25)
q75_mmd = all_islands["diag_mmd"].astype(float).quantile(0.75)
print(f"Global median MMD:                    {median_mmd_all:>8.4f}")
print(f"Global IQR:                           {q25_mmd:.4f} – {q75_mmd:.4f}")
print(f"Global mean MMD:                      {mean_mmd_all:>8.4f}")

print(f"\n--- Partial conservation ---")
partial = cores_per_gene[cores_per_gene["max_sp"] < len(PHYLO_ORDER)]
full = cores_per_gene[cores_per_gene["max_sp"] == len(PHYLO_ORDER)]
print(f"Genes with ALL cores in all species:   {len(full):>7,}")
print(f"Genes with partial conservation:       {len(partial):>7,}")
print(f"  (some cores match, others don't)")

print(f"\n{'=' * 70}")
print("Copy-paste numbers for the manuscript:")
print(f"  • ~{lnc_frac:.0f}% of lncRNAs contain ≥1 stability island")
print(f"  • median {lnc_median:.0f} islands per transcript")
print(f"  • {n_multi/n_total_cores:.0%} of cores found in ≥2 species")
print(f"  • {n_half/n_total_cores:.0%} of cores found in ≥5 species")
print(f"  • {n_all/n_total_cores:.0%} of cores found in all {len(PHYLO_ORDER)} species")
print(f"  • Median MMD = {median_mmd_all:.3f} (IQR {q25_mmd:.3f}–{q75_mmd:.3f})")
